In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# ── This is the most practical experiment ─────────────────
# chunk_size affects RAG quality directly
# Too small → chunks lose context → bad answers
# Too large → too much noise → bad answers
# Sweet spot → depends on your content

sample_text = """
LangChain is a framework for building applications with large language models.
It was created by Harrison Chase and first released in October 2022.
The framework provides a standard interface for chains, agents, and memory.

RAG or Retrieval Augmented Generation was introduced to solve hallucination problems.
In RAG systems, documents are split into chunks and stored in vector databases.
When a question is asked, relevant chunks are retrieved and sent to the LLM.
The LLM uses these chunks as context to generate accurate grounded answers.

LangGraph extends LangChain for building stateful multi-agent workflows.
It was released in 2024 as the recommended way to build production AI agents.
LangGraph supports human-in-the-loop, checkpointing, and multi-agent coordination.

Vector databases like ChromaDB and FAISS store document embeddings.
Embeddings are numerical representations of text meaning.
Similar meaning = similar numbers = close together in vector space.
Cosine similarity is used to find the most relevant chunks for a query.
""".strip()

print("═" * 50)
print("Chunk Size Experiment")
print("Same text — different chunk sizes — see the difference")
print("═" * 50)
print(f"Original text: {len(sample_text)} characters")
print()

# Test 4 different chunk sizes
configs = [
    {"chunk_size": 100,  "chunk_overlap": 10,  "label": "Too small"},
    {"chunk_size": 300,  "chunk_overlap": 30,  "label": "Small"},
    {"chunk_size": 500,  "chunk_overlap": 50,  "label": "Medium (sweet spot)"},
    {"chunk_size": 1000, "chunk_overlap": 100, "label": "Large"},
]

for config in configs:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=config["chunk_size"],
        chunk_overlap=config["chunk_overlap"]
    )
    chunks = splitter.split_text(sample_text)

    print(f"── {config['label']} (size={config['chunk_size']}) ──")
    print(f"Total chunks    : {len(chunks)}")
    print(f"Avg chunk size  : {sum(len(c) for c in chunks)//len(chunks)} chars")
    print(f"First chunk     : {chunks[0][:100]}...")
    print()

# ── The tradeoff explained ────────────────────────────────
print("═" * 50)
print("The Tradeoff")
print("═" * 50)
print("""
chunk_size=100  (too small)
  ✅ Very specific retrieval
  ❌ Loses context — half sentences, incomplete ideas
  ❌ Need to retrieve many chunks to get full answer

chunk_size=300  (small)
  ✅ Good specificity
  ✅ Decent context
  ❌ May still split related sentences

chunk_size=500  (sweet spot for most cases)
  ✅ Good balance of specificity + context
  ✅ Works well with most embedding models
  ✅ Start here for any new RAG project

chunk_size=1000 (large)
  ✅ Rich context per chunk
  ❌ Less specific — retrieves too much noise
  ❌ Expensive — more tokens sent to LLM
""")

# ── overlap experiment ────────────────────────────────────
print("═" * 50)
print("Overlap Experiment (chunk_size=500 fixed)")
print("═" * 50)

overlaps = [0, 50, 100, 200]
for overlap in overlaps:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=overlap
    )
    chunks = splitter.split_text(sample_text)
    print(f"overlap={overlap:3d} → {len(chunks)} chunks | "
          f"redundancy={(overlap/500*100):.0f}%")

print()
print("Rule of thumb: overlap = 10-20% of chunk_size")
print("chunk_size=500 → overlap=50-100")
print("Too much overlap = redundant content sent to LLM")
print("Too little overlap = ideas get cut at boundaries")

# ── Production defaults ───────────────────────────────────
print()
print("═" * 50)
print("Production defaults to remember")
print("═" * 50)
print("General text (PDFs, articles)  : chunk_size=500,  overlap=50")
print("Code                           : chunk_size=1000, overlap=100")
print("Short QA pairs                 : chunk_size=200,  overlap=20")
print("Always experiment on your data — no universal answer")

/opt/anaconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


══════════════════════════════════════════════════
Chunk Size Experiment
Same text — different chunk sizes — see the difference
══════════════════════════════════════════════════
Original text: 1045 characters

── Too small (size=100) ──
Total chunks    : 14
Avg chunk size  : 73 chars
First chunk     : LangChain is a framework for building applications with large language models....

── Small (size=300) ──
Total chunks    : 5
Avg chunk size  : 207 chars
First chunk     : LangChain is a framework for building applications with large language models.
It was created by Har...

── Medium (sweet spot) (size=500) ──
Total chunks    : 4
Avg chunk size  : 259 chars
First chunk     : LangChain is a framework for building applications with large language models.
It was created by Har...

── Large (size=1000) ──
Total chunks    : 2
Avg chunk size  : 521 chars
First chunk     : LangChain is a framework for building applications with large language models.
It was created by Har...

════════════════